<a href="https://colab.research.google.com/github/kej534923-maker/ECON5200-Applied-Data-Analytics/blob/main/lab_ch23_diagnostic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 23: FedSpeak 2.0 — Diagnostic Lab (ECON 5200)
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 40 min Core + 20 min Extension

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Topics:** Text preprocessing, TF-IDF vectorization, dictionary-based sentiment (LM vs Harvard GI), sentence-transformers embeddings, sentiment prediction evaluation.

**Verification checkpoints** are provided so you can confirm you found the right error.

**Time estimate:** ~60 minutes

---

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Install required packages and import libraries
# -----------------------------------------------------------
!pip install datasets nltk scikit-learn sentence-transformers -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report, roc_auc_score

from datasets import load_dataset

np.random.seed(42)

print('Libraries loaded. Ready to diagnose.')

---

## Part 1: DIAGNOSE — Find 3 Errors in This NLP Pipeline

The code below attempts to build a sentiment analysis pipeline for FOMC minutes.
There are **three deliberate errors** spread across three code cells. Each error
is a different type of NLP mistake:

1. A **tokenization/preprocessing** error
2. A **dictionary selection** error (wrong sentiment dictionary for the domain)
3. A **feature engineering** error in the TF-IDF configuration

**Your task:** Run each cell, identify the error, explain why it matters,
and fix it in Part 2.

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error. Find it.
# Step 1: Load and preprocess FOMC minutes
# -----------------------------------------------------------

# Load FOMC dataset
ds = load_dataset('vtasca/fomc-statements-minutes', split='train')
fomc_df = pd.DataFrame(ds)
fomc_df = fomc_df[fomc_df['Type'] == 'Minute'].copy()
fomc_df.rename(columns={'Text': 'text', 'Date': 'date'}, inplace=True)
fomc_df['date'] = pd.to_datetime(fomc_df['date'])
fomc_df = fomc_df.sort_values('date').reset_index(drop=True)
fomc_df['year'] = fomc_df['date'].dt.year

# ERROR: This tokenizer splits on whitespace only — no handling of
# punctuation, contractions, or special characters. "don't" stays as
# one token, "U.S." becomes "U.S." with trailing period, and
# "inflation-adjusted" stays hyphenated instead of splitting.
# A proper NLP tokenizer (nltk.word_tokenize) handles these cases.

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def bad_preprocess(text):
    """Preprocessing with a naive tokenizer."""
    text = text.lower()
    # BAD: split() instead of word_tokenize() — misses punctuation handling
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

fomc_df['clean_text'] = fomc_df['text'].apply(bad_preprocess)

# Check: many tokens will still have punctuation attached
sample_tokens = fomc_df['clean_text'].iloc[0].split()[:20]
print('Sample tokens from first document:')
print(sample_tokens)
print()
punct_tokens = [t for t in fomc_df['clean_text'].iloc[0].split() if not t.isalpha()]
print(f'Tokens containing non-alpha characters: {len(punct_tokens)}')
print(f'Examples: {punct_tokens[:10]}')
print()
print('PROBLEM: Many tokens still have attached punctuation.')
print('This means "rates," and "rates" are treated as different features.')

Tokenization error: Using split() leaves punctuation and malformed tokens in the corpus, causing noisy features.

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error.
# Step 2: Compute sentiment using WRONG dictionary
# -----------------------------------------------------------

# ERROR: Using a generic Harvard General Inquirer (GI) dictionary instead of
# the Loughran-McDonald (LM) dictionary designed for financial text.
# GI classifies "liability", "tax", "cost", "capital" as negative,
# but these are neutral in financial/economic contexts.

# Simplified Harvard GI negative words — includes many false positives for financial text
gi_negative = set([
    'abandon', 'adverse', 'against', 'bad', 'blame', 'capital', 'concern',
    'cost', 'costly', 'crisis', 'danger', 'debt', 'decline', 'deficit',
    'difficult', 'expense', 'fail', 'failure', 'fear', 'liability',
    'limit', 'limitation', 'loss', 'negative', 'obligation', 'penalty',
    'problem', 'risk', 'tax', 'threat', 'trouble', 'uncertain',
    'unemployment', 'volatile', 'weak', 'worse'
])

gi_positive = set([
    'achieve', 'advantage', 'benefit', 'confidence', 'gain', 'good',
    'growth', 'improve', 'increase', 'opportunity', 'positive', 'profit',
    'progress', 'strong', 'success', 'value'
])

def compute_gi_sentiment(text, neg_words, pos_words):
    """Compute sentiment using Harvard GI (wrong for financial text)."""
    tokens = text.lower().split()
    total = len(tokens)
    if total == 0:
        return {'net_sentiment': 0, 'neg_count': 0, 'pos_count': 0, 'neg_ratio': 0}

    neg_count = sum(1 for t in tokens if t in neg_words)
    pos_count = sum(1 for t in tokens if t in pos_words)

    return {
        'net_sentiment': (pos_count - neg_count) / total,
        'neg_count': neg_count,
        'pos_count': pos_count,
        'neg_ratio': neg_count / total
    }

gi_results = fomc_df['clean_text'].apply(
    lambda x: compute_gi_sentiment(x, gi_negative, gi_positive)
)
gi_df = pd.DataFrame(gi_results.tolist())

print('=== Harvard GI Sentiment (WRONG for financial text) ===')
print(f'Mean net sentiment: {gi_df["net_sentiment"].mean():.6f}')
print(f'Mean negative ratio: {gi_df["neg_ratio"].mean():.6f}')
print()

# Show the problem: count how many "negative" hits are false positives
false_positive_words = ['capital', 'cost', 'costly', 'debt', 'expense',
                        'liability', 'limit', 'limitation', 'obligation',
                        'penalty', 'tax']
sample_text = fomc_df['clean_text'].iloc[0].split()
fp_count = sum(1 for t in sample_text if t in false_positive_words)
total_neg = sum(1 for t in sample_text if t in gi_negative)
print(f'In first document: {fp_count} of {total_neg} "negative" words '
      f'are false positives ({fp_count/max(total_neg,1)*100:.0f}%)')
print('These are neutral financial terms misclassified by the GI dictionary.')

Dictionary error: The Harvard GI lexicon is inappropriate for financial text because it misclassifies neutral economic terms as negative. The Loughran–McDonald dictionary is more suitable for FOMC minutes.

In [ ]:
# -----------------------------------------------------------
# DIAGNOSE: This code has an error.
# Step 3: Build TF-IDF matrix with bad parameters
# -----------------------------------------------------------

# ERROR: max_df=1.0 means NO words are filtered by document frequency.
# Words like "the", "committee", "meeting" appear in every single document
# and dominate the TF-IDF matrix without contributing discriminating power.
# Also min_df=1 keeps every typo and OCR error.

bad_tfidf = TfidfVectorizer(
    min_df=1,          # Keep ALL words, even those in just 1 document (noise)
    max_df=1.0,        # Keep ALL words, even those in 100% of documents (no filtering)
    max_features=10000, # Large vocabulary with lots of noise
    ngram_range=(1, 1)  # Only unigrams — misses important bigrams like "interest rate"
)

bad_matrix = bad_tfidf.fit_transform(fomc_df['clean_text'])
feature_names = bad_tfidf.get_feature_names_out()

print(f'TF-IDF matrix shape: {bad_matrix.shape}')
print(f'Sparsity: {1 - bad_matrix.nnz / (bad_matrix.shape[0] * bad_matrix.shape[1]):.1%}')

# Show top terms — likely dominated by stop words and ubiquitous terms
mean_tfidf = np.asarray(bad_matrix.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[-15:][::-1]
print('\nTop 15 terms by average TF-IDF:')
for i in top_idx:
    doc_freq = (bad_matrix[:, i].toarray() > 0).sum()
    print(f'  {feature_names[i]:25s} avg_tfidf={mean_tfidf[i]:.4f}  '
          f'appears in {doc_freq}/{bad_matrix.shape[0]} docs')

print('\nPROBLEM: Many top terms appear in nearly ALL documents.')
print('These are background words, not discriminating features.')

TF-IDF error: The original vectorizer keeps both extremely rare and extremely common words and excludes bigrams, reducing feature quality and discriminating power.

---

## Part 2: FIX — Correct the Pipeline

Now write the **correct** NLP pipeline from scratch, fixing all three errors:

1. **Tokenization:** Use `nltk.word_tokenize()` + regex to strip non-alpha characters
2. **Dictionary:** Use Loughran-McDonald word lists instead of Harvard GI
3. **TF-IDF:** Set proper `min_df`, `max_df`, and include bigrams

**Verification checkpoints:**
- After fixing tokenization: zero tokens should contain non-alpha characters
- After switching to LM: false positive rate should drop below 10%
- After fixing TF-IDF: top terms should NOT include words appearing in >80% of documents

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Write the corrected NLP pipeline
# Fix all three errors from Part 1
# -----------------------------------------------------------

# Fix 1: Proper preprocessing with word_tokenize + regex cleaning
# YOUR CODE HERE


# Fix 2: Loughran-McDonald dictionary instead of Harvard GI
# YOUR CODE HERE


# Fix 3: Proper TF-IDF parameters (min_df=5, max_df=0.85, bigrams)
# YOUR CODE HERE


# VERIFICATION
# print('Fix 1 check — non-alpha tokens:', ...)
# print('Fix 2 check — false positive rate:', ...)
# print('Fix 3 check — top terms doc frequency:', ...)

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Write the corrected NLP pipeline
# Fix all three errors from Part 1
# -----------------------------------------------------------

import re
import numpy as np
import pandas as pd

from datasets import load_dataset

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# -----------------------------------------------------------
# Load FOMC minutes
# -----------------------------------------------------------

ds = load_dataset('vtasca/fomc-statements-minutes', split='train')
fomc_df = pd.DataFrame(ds)

fomc_df = fomc_df[fomc_df['Type'] == 'Minute'].copy()
fomc_df.rename(columns={'Text': 'text', 'Date': 'date'}, inplace=True)
fomc_df['date'] = pd.to_datetime(fomc_df['date'])
fomc_df = fomc_df.sort_values('date').reset_index(drop=True)
fomc_df['year'] = fomc_df['date'].dt.year

# -----------------------------------------------------------
# Fix 1: Proper preprocessing with word_tokenize + regex cleaning
# -----------------------------------------------------------

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def good_preprocess(text):
    """
    Proper preprocessing:
    1. lowercase
    2. tokenize with nltk.word_tokenize
    3. remove non-alpha characters with regex
    4. remove stopwords and short tokens
    5. lemmatize
    """
    text = str(text).lower()
    tokens = word_tokenize(text)

    cleaned_tokens = []
    for tok in tokens:
        # remove all non-alphabetic characters
        tok = re.sub(r'[^a-z]', '', tok)
        if tok and tok not in stop_words and len(tok) > 2:
            tok = lemmatizer.lemmatize(tok)
            cleaned_tokens.append(tok)

    return ' '.join(cleaned_tokens)

fomc_df['clean_text'] = fomc_df['text'].apply(good_preprocess)

# -----------------------------------------------------------
# Fix 2: Loughran-McDonald dictionary instead of Harvard GI
# -----------------------------------------------------------
# Note:
# In many class assignments, LM word lists are provided as text files.
# If your instructor gave you LM_Positive.txt and LM_Negative.txt,
# place them in the working directory and this code will load them.
# If not, a fallback mini LM-style dictionary is provided below so the
# notebook can still run.

def load_word_list(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return {line.strip().lower() for line in f if line.strip()}

try:
    lm_positive = load_word_list('LM_Positive.txt')
    lm_negative = load_word_list('LM_Negative.txt')
    print("Loaded full Loughran-McDonald dictionaries from local files.")
except FileNotFoundError:
    print("LM dictionary files not found. Using fallback mini LM-style dictionary.")

    # Fallback smaller LM-style lists for demonstration
    # These avoid obvious GI-style financial false positives like:
    # capital, tax, debt, liability, cost, obligation
    lm_negative = {
        'abandon', 'adverse', 'crisis', 'decline', 'deteriorate', 'difficult',
        'failure', 'fear', 'fragile', 'loss', 'negative', 'recession',
        'risk', 'risks', 'uncertain', 'uncertainty', 'volatile', 'weak', 'worse'
    }

    lm_positive = {
        'achieve', 'benefit', 'confidence', 'gain', 'growth', 'improve',
        'improved', 'improvement', 'positive', 'progress', 'resilient',
        'strong', 'strengthen', 'stability', 'stable', 'success'
    }

def compute_lm_sentiment(text, neg_words, pos_words):
    """
    Compute sentiment using the Loughran-McDonald dictionary.
    """
    tokens = text.split()
    total = len(tokens)

    if total == 0:
        return {
            'net_sentiment': 0.0,
            'neg_count': 0,
            'pos_count': 0,
            'neg_ratio': 0.0
        }

    neg_count = sum(1 for t in tokens if t in neg_words)
    pos_count = sum(1 for t in tokens if t in pos_words)

    return {
        'net_sentiment': (pos_count - neg_count) / total,
        'neg_count': neg_count,
        'pos_count': pos_count,
        'neg_ratio': neg_count / total
    }

lm_results = fomc_df['clean_text'].apply(
    lambda x: compute_lm_sentiment(x, lm_negative, lm_positive)
)
lm_df = pd.DataFrame(lm_results.tolist())

# -----------------------------------------------------------
# Fix 3: Proper TF-IDF parameters (min_df=5, max_df=0.85, bigrams)
# -----------------------------------------------------------

good_tfidf = TfidfVectorizer(
    min_df=5,
    max_df=0.80,
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = good_tfidf.fit_transform(fomc_df['clean_text'])
feature_names = good_tfidf.get_feature_names_out()

# -----------------------------------------------------------
# VERIFICATION
# -----------------------------------------------------------

# Fix 1 check — non-alpha tokens
all_tokens = ' '.join(fomc_df['clean_text']).split()
non_alpha_tokens = [t for t in all_tokens if not t.isalpha()]
print('Fix 1 check — non-alpha tokens:', len(non_alpha_tokens))

# Fix 2 check — false positive rate
# These are words that GI often falsely labels as negative in financial text
false_positive_words = {
    'capital', 'cost', 'costly', 'debt', 'expense',
    'liability', 'limit', 'limitation', 'obligation',
    'penalty', 'tax'
}

all_lm_negative_hits = []
for doc in fomc_df['clean_text']:
    toks = doc.split()
    all_lm_negative_hits.extend([t for t in toks if t in lm_negative])

fp_count = sum(1 for t in all_lm_negative_hits if t in false_positive_words)
fp_rate = fp_count / max(len(all_lm_negative_hits), 1)
print('Fix 2 check — false positive rate:', round(fp_rate, 4))

# Fix 3 check — top terms doc frequency
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[-15:][::-1]

top_term_df = []
print('\nTop 15 TF-IDF terms and document frequency:')
for i in top_idx:
    term = feature_names[i]
    doc_freq = (tfidf_matrix[:, i].toarray() > 0).sum()
    doc_share = doc_freq / tfidf_matrix.shape[0]
    top_term_df.append((term, doc_freq, round(doc_share, 4)))
    print(f'{term:30s} appears in {doc_freq}/{tfidf_matrix.shape[0]} docs ({doc_share:.2%})')

print('\nFix 3 check — top terms doc frequency:', top_term_df)

# Optional assertions for checkpoints
assert len(non_alpha_tokens) == 0, "Fix 1 failed: found non-alpha tokens."
assert fp_rate < 0.10, "Fix 2 failed: false positive rate is >= 10%."
assert all(doc_share <= 0.80 for _, _, doc_share in top_term_df), \
    "Fix 3 failed: some top terms appear in >80% of documents."

print('\nAll checks passed.')
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Mean LM net sentiment: {lm_df["net_sentiment"].mean():.6f}')
print(f'Mean LM negative ratio: {lm_df["neg_ratio"].mean():.6f}')

---

## Part 3: EXTEND — Sentence-Transformers Embeddings

TF-IDF treats each word independently (bag-of-words). **Sentence-transformers**
encode entire sentences or documents as dense vectors that capture meaning,
context, and word order.

We will:
1. Encode FOMC documents with a pre-trained sentence-transformer
2. Cluster on embeddings and compare to TF-IDF clusters
3. Evaluate which representation better predicts Fed rate decisions

In [ ]:
# -----------------------------------------------------------
# GUIDED — Run as-is
# Step 3a: Encode FOMC documents with sentence-transformers
# -----------------------------------------------------------

from sentence_transformers import SentenceTransformer

# Use a lightweight model suitable for Colab
# all-MiniLM-L6-v2 produces 384-dimensional dense embeddings
st_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode documents (truncate long docs to first 512 tokens for speed)
# In production, you would use a chunking strategy for long documents
print('Encoding documents with sentence-transformers...')
print('(This may take 2-5 minutes on CPU)')

# Use first 2000 characters of each document (model has 256 token limit)
truncated_texts = fomc_df['text'].str[:2000].tolist()
embeddings = st_model.encode(truncated_texts, show_progress_bar=True, batch_size=16)

print(f'\nEmbedding matrix shape: {embeddings.shape}')
print(f'  → {embeddings.shape[0]} documents × {embeddings.shape[1]} dimensions')
print(f'Density: 100% (dense vectors, unlike sparse TF-IDF)')

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Cluster on embeddings and compare to TF-IDF
# -----------------------------------------------------------

# Step A: K-Means on sentence-transformer embeddings (K=3)
kmeans_emb = KMeans(n_clusters=3, random_state=42, n_init=10)
fomc_df['cluster_emb'] = kmeans_emb.fit_predict(embeddings)

# Step B: K-Means on TF-IDF (use your corrected TF-IDF from Part 2)
# If you haven't fixed Part 2 yet, use these default parameters:
# tfidf_corrected = TfidfVectorizer(min_df=5, max_df=0.85, max_features=5000, ngram_range=(1,2))
# tfidf_matrix_corrected = tfidf_corrected.fit_transform(fomc_df['clean_text'])
# -----------------------------------------------------------
# Step B: TF-IDF clustering (with dimensionality reduction)
# -----------------------------------------------------------

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=50, random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)

kmeans_tfidf = KMeans(n_clusters=3, random_state=42, n_init=10)
fomc_df['cluster_tfidf'] = kmeans_tfidf.fit_predict(tfidf_reduced)

print("TF-IDF reduced shape:", tfidf_reduced.shape)

# YOUR CODE: Reduce TF-IDF to 50 dims with TruncatedSVD, then cluster
# svd = TruncatedSVD(n_components=50, random_state=42)
# tfidf_reduced = svd.fit_transform(tfidf_matrix_corrected)
# kmeans_tfidf = KMeans(n_clusters=3, random_state=42, n_init=10)
# fomc_df['cluster_tfidf'] = kmeans_tfidf.fit_predict(tfidf_reduced)
# -----------------------------------------------------------
# Step C: Compare clustering quality
# -----------------------------------------------------------

from sklearn.metrics import silhouette_score

sil_emb = silhouette_score(embeddings, fomc_df['cluster_emb'])
sil_tfidf = silhouette_score(tfidf_reduced, fomc_df['cluster_tfidf'])

print(f'Silhouette — Embeddings: {sil_emb:.3f}')
print(f'Silhouette — TF-IDF:     {sil_tfidf:.3f}')

# Step D: Visualize both clusterings side by side in PCA space
# -----------------------------------------------------------
# Step D: PCA visualization
# -----------------------------------------------------------

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# PCA for embeddings
pca_emb = PCA(n_components=2, random_state=42)
emb_2d = pca_emb.fit_transform(embeddings)

# PCA for TF-IDF
pca_tfidf = PCA(n_components=2, random_state=42)
tfidf_2d = pca_tfidf.fit_transform(tfidf_reduced)

# Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Embedding clusters
axes[0].scatter(
    emb_2d[:, 0], emb_2d[:, 1],
    c=fomc_df['cluster_emb'],
    alpha=0.7
)
axes[0].set_title("Sentence-Transformer Clusters")

# TF-IDF clusters
axes[1].scatter(
    tfidf_2d[:, 0], tfidf_2d[:, 1],
    c=fomc_df['cluster_tfidf'],
    alpha=0.7
)
axes[1].set_title("TF-IDF Clusters")

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------
# YOUR TASK — Compare predictive power: TF-IDF vs Embeddings
# Predict whether the Fed raised rates at the NEXT meeting
# -----------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import TruncatedSVD

# -----------------------------------------------------------
# Step 1: Create synthetic binary target
# -----------------------------------------------------------

tightening_years = set([2004, 2005, 2006, 2015, 2016, 2017, 2018, 2022, 2023])
fomc_df['tightening'] = fomc_df['year'].isin(tightening_years).astype(int)

print(f'Tightening meetings: {fomc_df["tightening"].sum()}')
print(f'Easing/holding meetings: {(1 - fomc_df["tightening"]).sum()}')

# -----------------------------------------------------------
# Step 2: Prepare features
# Assumes:
#   tfidf_matrix  -> corrected TF-IDF matrix from Part 2
#   embeddings    -> sentence-transformer embeddings from Part 3
# -----------------------------------------------------------

svd = TruncatedSVD(n_components=50, random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)

X_tfidf = tfidf_reduced
X_emb = np.array(embeddings)
y = fomc_df['tightening'].values

print("TF-IDF reduced shape:", X_tfidf.shape)
print("Embeddings shape:", X_emb.shape)
print("Target shape:", y.shape)

# -----------------------------------------------------------
# Step 3: TimeSeriesSplit evaluation function
# -----------------------------------------------------------

def evaluate_timeseries_auc(X, y, model_name, n_splits=5):
    """
    Evaluate AUC-ROC using TimeSeriesSplit and Logistic Regression.
    Skips folds where train or test has only one class.
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    auc_scores = []

    print(f'\n=== {model_name} ===')

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        train_classes = np.unique(y_train)
        test_classes = np.unique(y_test)

        print(
            f'Fold {fold}: train={len(train_idx)}, test={len(test_idx)}, '
            f'train_classes={train_classes}, test_classes={test_classes}'
        )

        # training set must contain both classes
        if len(train_classes) < 2:
            print(f'  -> Skipped fold {fold}: training set has only one class.')
            continue

        # test set must contain both classes for AUC
        if len(test_classes) < 2:
            print(f'  -> Skipped fold {fold}: test set has only one class.')
            continue

        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        auc_scores.append(auc)

        print(f'  -> AUC={auc:.3f}')

    if len(auc_scores) > 0:
        print(f'Mean AUC-ROC ({model_name}): {np.mean(auc_scores):.3f}')
        print(f'Std AUC-ROC  ({model_name}): {np.std(auc_scores):.3f}')
    else:
        print(f'No valid folds available for {model_name}.')

    return auc_scores

# -----------------------------------------------------------
# Step 4: Run evaluation
# -----------------------------------------------------------

auc_tfidf = evaluate_timeseries_auc(X_tfidf, y, 'TF-IDF (50D SVD)', n_splits=5)
auc_emb = evaluate_timeseries_auc(X_emb, y, 'Sentence-Transformer Embeddings', n_splits=5)

# -----------------------------------------------------------
# Step 5: Final comparison
# -----------------------------------------------------------

mean_auc_tfidf = np.mean(auc_tfidf) if len(auc_tfidf) > 0 else np.nan
mean_auc_emb = np.mean(auc_emb) if len(auc_emb) > 0 else np.nan

print('\n=== Final Comparison ===')
print(f'Mean AUC-ROC — TF-IDF:     {mean_auc_tfidf:.3f}')
print(f'Mean AUC-ROC — Embeddings: {mean_auc_emb:.3f}')

if np.isnan(mean_auc_tfidf) or np.isnan(mean_auc_emb):
    print('Comparison incomplete because one representation had too few valid folds.')
elif mean_auc_emb > mean_auc_tfidf:
    print('Embeddings performed better in predicting tightening periods.')
elif mean_auc_emb < mean_auc_tfidf:
    print('TF-IDF performed better in predicting tightening periods.')
else:
    print('Both representations performed equally on average.')

# -----------------------------------------------------------
# Step 6: Store results
# -----------------------------------------------------------

max_len = max(len(auc_tfidf), len(auc_emb))
results_df = pd.DataFrame({
    'fold': list(range(1, max_len + 1)),
    'auc_tfidf': auc_tfidf + [np.nan] * (max_len - len(auc_tfidf)),
    'auc_emb': auc_emb + [np.nan] * (max_len - len(auc_emb)),
})

print('\nPer-fold results:')
print(results_df)

---

## Part 4: Module Output — `fomc_sentiment.py`

Write a reusable Python module for FOMC text analysis.
This is a **portfolio artifact** that demonstrates production-grade NLP work.

### Requirements

```python
# src/fomc_sentiment.py

def preprocess_fomc(text: str) -> str:
    """Clean and tokenize FOMC minutes text.
    
    Steps: lowercase, regex clean, word_tokenize, stop words, lemmatize.
    Returns space-joined clean tokens.
    """
    ...

def compute_lm_sentiment(text: str) -> dict:
    """Compute Loughran-McDonald sentiment scores.
    
    Returns dict with 'net_sentiment', 'uncertainty',
    'neg_count', 'pos_count', 'unc_count', 'total_words'.
    """
    ...

def build_tfidf_matrix(texts: list, min_df=5, max_df=0.85,
                       max_features=5000) -> tuple:
    """Build TF-IDF matrix from preprocessed texts.
    
    Returns (sparse_matrix, feature_names, vectorizer).
    """
    ...
```

In [ ]:
# %%writefile src/fomc_sentiment.py
"""
fomc_sentiment.py — FOMC Text Analysis Module

Reusable functions for preprocessing, sentiment scoring, and
TF-IDF vectorization of Federal Reserve meeting minutes.

Author: Jia Ke
Course: ECON 5200, Lab 23
"""

import re
from typing import Tuple, List, Dict, Any

import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer


# Loughran-McDonald word lists (simplified)
LM_NEGATIVE = set([
    'adverse', 'adversely', 'against', 'concern', 'concerned', 'concerns',
    'decline', 'declined', 'declining', 'decrease', 'decreased', 'deficit',
    'deteriorate', 'deteriorated', 'deteriorating', 'difficult', 'difficulty',
    'downturn', 'fail', 'failure', 'falling', 'loss', 'losses', 'negative',
    'negatively', 'recession', 'recessionary', 'risk', 'risks', 'risky',
    'severe', 'severely', 'slowdown', 'sluggish', 'stress', 'stressed',
    'threat', 'threaten', 'troubled', 'uncertain', 'uncertainty',
    'unfavorable', 'volatile', 'volatility', 'vulnerability', 'vulnerable',
    'weak', 'weaken', 'weakened', 'weakness', 'worse', 'worsen', 'worsened'
])

LM_POSITIVE = set([
    'achieve', 'achieved', 'achievement', 'benefit', 'beneficial', 'confidence',
    'confident', 'favorable', 'gain', 'gained', 'gains', 'good', 'growth',
    'improve', 'improved', 'improvement', 'improving', 'increase', 'increased',
    'opportunity', 'optimism', 'optimistic', 'positive', 'positively',
    'profit', 'profitable', 'progress', 'rebound', 'recover', 'recovery',
    'strength', 'strengthen', 'strong', 'stronger', 'success', 'successful'
])

LM_UNCERTAINTY = set([
    'approximate', 'approximately', 'assume', 'assumption', 'believe',
    'cautious', 'could', 'depend', 'depends', 'doubt', 'estimate',
    'expect', 'expected', 'forecast', 'indefinite', 'likelihood', 'may',
    'might', 'nearly', 'perhaps', 'possible', 'possibly', 'predict',
    'preliminary', 'probable', 'probably', 'risk', 'roughly', 'seem',
    'suggest', 'tentative', 'uncertain', 'uncertainty', 'unclear',
    'unpredictable', 'variable'
])


_STOP_WORDS = set(stopwords.words('english'))
_LEMMATIZER = WordNetLemmatizer()


def preprocess_fomc(text: str) -> str:
    """Clean and tokenize FOMC minutes text.

    Steps:
    1. lowercase
    2. word_tokenize
    3. regex clean to keep alphabetic characters only
    4. remove stop words
    5. lemmatize
    6. return space-joined tokens

    Parameters
    ----------
    text : str
        Raw FOMC text.

    Returns
    -------
    str
        Cleaned, tokenized, and lemmatized text.
    """
    if text is None:
        return ""

    if not isinstance(text, str):
        text = str(text)

    text = text.lower()
    raw_tokens = word_tokenize(text)

    clean_tokens = []
    for token in raw_tokens:
        token = re.sub(r'[^a-z]', '', token)
        if token and token not in _STOP_WORDS and len(token) > 2:
            token = _LEMMATIZER.lemmatize(token)
            if token:
                clean_tokens.append(token)

    return " ".join(clean_tokens)


def compute_lm_sentiment(text: str) -> Dict[str, Any]:
    """Compute Loughran-McDonald sentiment scores.

    Parameters
    ----------
    text : str
        Preprocessed or raw text. Best practice is to pass preprocessed text.

    Returns
    -------
    dict
        Dictionary with:
        - net_sentiment
        - uncertainty
        - neg_count
        - pos_count
        - unc_count
        - total_words
    """
    if text is None:
        text = ""

    if not isinstance(text, str):
        text = str(text)

    tokens = text.split()
    total_words = len(tokens)

    if total_words == 0:
        return {
            'net_sentiment': 0.0,
            'uncertainty': 0.0,
            'neg_count': 0,
            'pos_count': 0,
            'unc_count': 0,
            'total_words': 0
        }

    neg_count = sum(1 for token in tokens if token in LM_NEGATIVE)
    pos_count = sum(1 for token in tokens if token in LM_POSITIVE)
    unc_count = sum(1 for token in tokens if token in LM_UNCERTAINTY)

    net_sentiment = (pos_count - neg_count) / total_words
    uncertainty = unc_count / total_words

    return {
        'net_sentiment': net_sentiment,
        'uncertainty': uncertainty,
        'neg_count': neg_count,
        'pos_count': pos_count,
        'unc_count': unc_count,
        'total_words': total_words
    }


def build_tfidf_matrix(
    texts: List[str],
    min_df: int = 5,
    max_df: float = 0.85,
    max_features: int = 5000
) -> Tuple:
    """Build TF-IDF matrix from preprocessed texts.

    Parameters
    ----------
    texts : list of str
        Collection of preprocessed documents.
    min_df : int, default=5
        Minimum document frequency.
    max_df : float, default=0.85
        Maximum document frequency proportion.
    max_features : int, default=5000
        Maximum number of features.

    Returns
    -------
    tuple
        (sparse_matrix, feature_names, vectorizer)
    """
    if texts is None or len(texts) == 0:
        raise ValueError("texts must be a non-empty list of strings.")

    processed_texts = []
    for text in texts:
        if text is None:
            processed_texts.append("")
        elif isinstance(text, str):
            processed_texts.append(text)
        else:
            processed_texts.append(str(text))

    vectorizer = TfidfVectorizer(
        min_df=min_df,
        max_df=max_df,
        max_features=max_features,
        ngram_range=(1, 2)
    )

    sparse_matrix = vectorizer.fit_transform(processed_texts)
    feature_names = vectorizer.get_feature_names_out()

    return sparse_matrix, feature_names, vectorizer


# --- Quick self-test ---
if __name__ == '__main__':
    test_text = "The committee noted that inflation remained elevated above target."
    clean = preprocess_fomc(test_text)
    print(f'Preprocessed: {clean}')

    sentiment = compute_lm_sentiment(clean)
    print(f'Sentiment: {sentiment}')

    test_docs = [
        preprocess_fomc("Inflation remained elevated and risks persisted."),
        preprocess_fomc("Economic growth improved and confidence strengthened."),
        preprocess_fomc("The committee expressed uncertainty about future conditions."),
        preprocess_fomc("Labor market conditions remained solid and inflation slowed."),
        preprocess_fomc("Members noted downside risks and weaker activity."),
        preprocess_fomc("Growth and recovery continued despite uncertainty.")
    ]

    X, features, vec = build_tfidf_matrix(test_docs, min_df=1, max_df=1.0, max_features=20)
    print(f"TF-IDF shape: {X.shape}")
    print(f"First 10 features: {features[:10]}")
    print('fomc_sentiment.py loaded successfully.')

---

## Challenge: Compare TF-IDF vs Embedding Predictive Power

Build a proper expanding-window evaluation of both TF-IDF and embedding-based
sentiment for predicting Fed rate decisions. Use at least 5 splits.
Report mean AUC and standard deviation across folds.

Write a 1-paragraph summary of which representation is better and why.

In [ ]:
# -----------------------------------------------------------
# CHALLENGE — Full comparison of TF-IDF vs Embeddings
# -----------------------------------------------------------

import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# -----------------------------------------------------------
# 0. Target variable
# -----------------------------------------------------------

tightening_years = set([2004, 2005, 2006, 2015, 2016, 2017, 2018, 2022, 2023])
fomc_df = fomc_df.sort_values('date').reset_index(drop=True)
fomc_df['tightening'] = fomc_df['year'].isin(tightening_years).astype(int)

y = fomc_df['tightening'].values

print(f'Total observations: {len(fomc_df)}')
print(f'Tightening meetings: {fomc_df["tightening"].sum()}')
print(f'Easing/holding meetings: {(1 - fomc_df["tightening"]).sum()}')

# -----------------------------------------------------------
# 1. Prepare features
# Assumes:
#   tfidf_matrix  -> corrected TF-IDF matrix from Part 2
#   embeddings    -> sentence-transformer embeddings from Part 3
# -----------------------------------------------------------

# Reduce TF-IDF to 50 dimensions
svd = TruncatedSVD(n_components=50, random_state=42)
X_tfidf = svd.fit_transform(tfidf_matrix)

# Embeddings
X_emb = np.array(embeddings)

print("TF-IDF reduced shape:", X_tfidf.shape)
print("Embeddings shape:", X_emb.shape)

# -----------------------------------------------------------
# 2. Create expanding-window splits
# At least 5 splits
# -----------------------------------------------------------

def make_expanding_window_splits(n_samples, n_splits=5, initial_train_size=80):
    """
    Create expanding-window train/test splits.

    Parameters
    ----------
    n_samples : int
        Total number of observations.
    n_splits : int
        Number of splits.
    initial_train_size : int
        Initial training window size.

    Returns
    -------
    list of tuples
        Each tuple is (train_idx, test_idx).
    """
    if initial_train_size >= n_samples:
        raise ValueError("initial_train_size must be smaller than n_samples.")

    remaining = n_samples - initial_train_size
    test_size = remaining // n_splits

    if test_size < 1:
        raise ValueError("Not enough data to create the requested number of splits.")

    splits = []
    train_end = initial_train_size

    for i in range(n_splits):
        test_start = train_end
        if i < n_splits - 1:
            test_end = test_start + test_size
        else:
            test_end = n_samples  # last split takes the remainder

        train_idx = np.arange(0, train_end)
        test_idx = np.arange(test_start, test_end)

        if len(test_idx) > 0:
            splits.append((train_idx, test_idx))

        train_end = test_end

    return splits

splits = make_expanding_window_splits(
    n_samples=len(y),
    n_splits=5,
    initial_train_size=80
)

print("\nExpanding-window splits:")
for i, (train_idx, test_idx) in enumerate(splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

# -----------------------------------------------------------
# 3. Evaluation function
# -----------------------------------------------------------

def evaluate_expanding_auc(X, y, splits, model_name):
    """
    Evaluate expanding-window AUC-ROC using Logistic Regression.
    Skips folds where train/test has only one class.
    """
    auc_scores = []

    print(f'\n=== {model_name} ===')

    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        train_classes = np.unique(y_train)
        test_classes = np.unique(y_test)

        print(
            f'Fold {fold}: '
            f'train={len(train_idx)}, test={len(test_idx)}, '
            f'train_classes={train_classes}, test_classes={test_classes}'
        )

        if len(train_classes) < 2:
            print(f'  -> Skipped fold {fold}: training set has only one class.')
            continue

        if len(test_classes) < 2:
            print(f'  -> Skipped fold {fold}: test set has only one class.')
            continue

        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        auc_scores.append(auc)

        print(f'  -> AUC={auc:.3f}')

    return auc_scores

# -----------------------------------------------------------
# 4. Run evaluation
# -----------------------------------------------------------

auc_tfidf = evaluate_expanding_auc(X_tfidf, y, splits, 'TF-IDF (50D SVD)')
auc_emb = evaluate_expanding_auc(X_emb, y, splits, 'Sentence-Transformer Embeddings')

# -----------------------------------------------------------
# 5. Summary stats
# -----------------------------------------------------------

mean_tfidf = np.mean(auc_tfidf) if len(auc_tfidf) > 0 else np.nan
std_tfidf = np.std(auc_tfidf) if len(auc_tfidf) > 0 else np.nan

mean_emb = np.mean(auc_emb) if len(auc_emb) > 0 else np.nan
std_emb = np.std(auc_emb) if len(auc_emb) > 0 else np.nan

print("\nExpected output format:")
print(f"TF-IDF AUC:      {mean_tfidf:.2f} ± {std_tfidf:.2f} (mean ± std across valid folds)")
print(f"Embeddings AUC:  {mean_emb:.2f} ± {std_emb:.2f}")

if np.isnan(mean_tfidf) and np.isnan(mean_emb):
    winner = "None"
elif np.isnan(mean_tfidf):
    winner = "Embeddings"
elif np.isnan(mean_emb):
    winner = "TF-IDF"
elif mean_emb > mean_tfidf:
    winner = "Embeddings"
elif mean_tfidf > mean_emb:
    winner = "TF-IDF"
else:
    winner = "Tie"

print(f"Winner: {winner}")

# -----------------------------------------------------------
# 6. Per-fold results table
# -----------------------------------------------------------

max_len = max(len(auc_tfidf), len(auc_emb))
results_df = pd.DataFrame({
    'fold': list(range(1, max_len + 1)),
    'auc_tfidf': auc_tfidf + [np.nan] * (max_len - len(auc_tfidf)),
    'auc_emb': auc_emb + [np.nan] * (max_len - len(auc_emb)),
})

print("\nPer-fold results:")
print(results_df)

# -----------------------------------------------------------
# 7. 1-paragraph interpretation
# -----------------------------------------------------------

if winner == "Embeddings":
    interpretation = (
        f"Using an expanding-window evaluation, sentence-transformer embeddings "
        f"outperformed TF-IDF for predicting tightening periods, with an average "
        f"AUC of {mean_emb:.2f} compared with {mean_tfidf:.2f} for TF-IDF. "
        f"This suggests that contextual embeddings capture richer semantic information "
        f"from FOMC minutes, including tone, policy nuance, and meaning across phrases, "
        f"whereas TF-IDF relies mainly on sparse word-frequency patterns. Although TF-IDF "
        f"can still capture useful signals, the stronger performance of embeddings indicates "
        f"that semantic representations are better suited for policy-text prediction tasks."
    )
elif winner == "TF-IDF":
    interpretation = (
        f"Using an expanding-window evaluation, TF-IDF outperformed sentence-transformer "
        f"embeddings for predicting tightening periods, with an average AUC of "
        f"{mean_tfidf:.2f} compared with {mean_emb:.2f} for embeddings. This indicates "
        f"that in this dataset, sparse term-frequency features may align more directly with "
        f"policy-related language shifts than dense semantic embeddings. A likely reason is "
        f"that specific keywords and repeated policy phrases in FOMC minutes are highly informative, "
        f"allowing TF-IDF to separate tightening from non-tightening regimes effectively."
    )
else:
    interpretation = (
        f"Using an expanding-window evaluation, TF-IDF and sentence-transformer embeddings "
        f"performed similarly in predicting tightening periods, with average AUC values of "
        f"{mean_tfidf:.2f} and {mean_emb:.2f}, respectively. This suggests that both sparse "
        f"term-frequency information and dense semantic representations capture meaningful signals "
        f"from FOMC minutes. TF-IDF may benefit from explicit policy vocabulary, while embeddings "
        f"benefit from contextual meaning, so the best choice may depend on whether interpretability "
        f"or semantic richness is more important for the downstream task."
    )

print("\n1-paragraph interpretation:")
print(interpretation)

Using an expanding-window evaluation, sentence-transformer embeddings outperformed TF-IDF in predicting tightening periods. This suggests that embeddings capture richer semantic and contextual information from FOMC minutes, including policy tone and meaning across phrases, while TF-IDF mainly relies on sparse word-frequency patterns. Although TF-IDF remains interpretable and useful, the stronger AUC of embeddings indicates that contextual representations are better suited for this policy-text prediction task.

---

## Digital Portfolio: Institutional Signaling

### Generate Your Professional README

Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.**

```text
"I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed and fixed a broken NLP pipeline (naive tokenizer, wrong sentiment
  dictionary, bad TF-IDF parameters)
* Corrected preprocessing with nltk.word_tokenize, switched from Harvard GI to
  Loughran-McDonald dictionary, fixed TF-IDF min_df/max_df
* Encoded FOMC documents with sentence-transformers (all-MiniLM-L6-v2)
* Compared TF-IDF vs embedding-based clustering and predictive power
* Built a reusable fomc_sentiment.py module with preprocess_fomc(),
  compute_lm_sentiment(), and build_tfidf_matrix()
* Key finding: [TF-IDF/Embeddings] achieved higher AUC ([VALUE]) for
  predicting Fed rate decisions

**Please write a README.md entry including:**
1. Project Title: FedSpeak 2.0 — NLP Pipeline for Central Bank Communications
2. Objective: A professional one-sentence summary
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Make this sound like a professional tech economist wrote it."
```

### Push to GitHub

```bash
cd econ-lab-23-fedspeak
git add notebooks/ src/ figures/ README.md
git commit -m "Lab 23: FedSpeak 2.0 — NLP Pipeline, Embeddings & Prediction"
git push origin main
```

Submit your GitHub repo link on Canvas.